In [ ]:
# Step 1: Import required libraries

import pandas as pd
from pathlib import Path

print("Libraries imported successfully!")

In [ ]:
from pathlib import Path

# Step 1: Define project directory
PROJECT_DIR = Path.cwd()

# Step 2: Define raw data path
RAW_PATH = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "corporate_emissions_dirty_v1.csv"
)

# Step 3: Define output path
OUTPUT_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "corporate_emissions_clean.csv"
)

# Step 4: Check paths
print("Project directory:", PROJECT_DIR)
print("Raw data exists:", RAW_PATH.is_file())
print("Output path:", OUTPUT_PATH)

In [ ]:
# Step 2.2: Standardize Scope values

def clean_scope(data):

    cleaned = data.copy()

    cleaned["Scope"] = (
        cleaned["Scope"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace("scope", "", regex=False)
        .str.strip()
    )

    # Check for unexpected values
    valid_scopes = ["1", "2", "3"]

    if not cleaned["Scope"].isin(valid_scopes).all():
        raise ValueError("Invalid Scope values found")

    return cleaned




In [ ]:
# Step 2.3.2: Fill missing emissions

def fill_missing_emissions(data):

    cleaned = data.copy()

    # Find rows with missing emissions
    missing = cleaned["Emissions tCO2e"].isna()

    # Check required input data
    required = ["Activity", "Emission Factor"]

    if cleaned.loc[missing, required].isna().any().any():
        raise ValueError(
            "Missing Activity or Emission Factor"
        )

    # Calculate emissions only for missing rows
    cleaned.loc[missing, "Emissions tCO2e"] = (
        cleaned.loc[missing, "Activity"]
        * cleaned.loc[missing, "Emission Factor"]
    )

    return cleaned



In [ ]:
# Step 2.4: Remove duplicate records

def remove_duplicates(data):

    cleaned = data.copy()

    # Count duplicate rows
    n_duplicates = cleaned.duplicated().sum()

    print(f"Duplicates found: {n_duplicates}")

    # Remove duplicate rows
    cleaned = cleaned.drop_duplicates()

    # Reset row index
    cleaned = cleaned.reset_index(drop=True)

    return cleaned




In [ ]:
# Step 4: Build a complete cleaning pipeline

def run_pipeline(raw_path, output_path):

    # 1. Read raw data
    data = pd.read_csv(raw_path)

    print("Raw data shape:", data.shape)

    # 2. Apply cleaning functions
    data = clean_scope(data)

    data = fill_missing_emissions(data)

    data = remove_duplicates(data)

    # 3. Validate cleaned data
    numeric_cols = [
        "Year",
        "Activity",
        "Emission Factor",
        "Emissions tCO2e"
    ]

    for col in numeric_cols:
        if not pd.api.types.is_numeric_dtype(data[col]):
            raise TypeError(f"{col} must be numeric")

    if data["Emissions tCO2e"].isna().any():
        raise ValueError("Missing emissions remain")

    if not data["Scope"].isin(["1", "2", "3"]).all():
        raise ValueError("Invalid Scope found")

    if data.duplicated().any():
        raise ValueError("Duplicate rows remain")

    # 4. Save processed data
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    data.to_csv(output_path, index=False)

    print("Processed data shape:", data.shape)
    print("Saved to:", output_path)

    return data

In [ ]:
# Run the complete pipeline

df_processed = run_pipeline(
    RAW_PATH,
    OUTPUT_PATH
)

display(df_processed.head())